In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import pandas as pd
import time
import os
from pathlib import Path

def scrape_football_stats(output_dir=None):
    # Setup output directory
    if output_dir is None:
        # Use current user's documents folder if no directory specified
        output_dir = str(Path.home() / "Documents")
    
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)
    
    # Create full file path
    csv_filename = os.path.join(output_dir, "football_stats_big5_expanded.csv")
    
    # Setup Chrome driver with additional options
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')
    options.add_argument('--disable-gpu')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--window-size=1920,1080')
    driver = webdriver.Chrome(options=options)
    
    # URL of the page
    url = "https://fbref.com/en/comps/Big5/stats/squads/Big-5-European-Leagues-Stats"
    
    try:
        # Navigate to the page
        driver.get(url)
        print("Page loaded successfully")
        
        # Wait for the page to load completely
        wait = WebDriverWait(driver, 20)
        
        try:
            # Find the table container
            print("Looking for table container...")
            table_container = wait.until(EC.presence_of_element_located((
                By.CSS_SELECTOR, 
                "div#div_stats_teams_standard_for"
            )))
            print("Table container found")

            # Find and click the switcher
            print("Looking for switcher...")
            switcher = wait.until(EC.element_to_be_clickable((
                By.CSS_SELECTOR, 
                "div.switcher"
            )))
            driver.execute_script("arguments[0].scrollIntoView(true);", switcher)
            time.sleep(1)
            driver.execute_script("arguments[0].click();", switcher)
            print("Successfully clicked switcher")

            # Wait for table to update
            print("Waiting for table to load...")
            time.sleep(3)
            
            table = wait.until(EC.presence_of_element_located((
                By.CSS_SELECTOR, 
                "table#stats_teams_standard_for"
            )))
            print("Table found")

            # Get headers
            print("Getting headers...")
            header_rows = table.find_elements(By.TAG_NAME, "thead")[0].find_elements(By.TAG_NAME, "tr")
            header_cells = header_rows[1].find_elements(By.TAG_NAME, "th")
            headers = [cell.get_attribute('data-stat') for cell in header_cells]
            print(f"Found {len(headers)} headers")

            # Get data rows
            print("Getting data rows...")
            tbody = table.find_element(By.TAG_NAME, "tbody")
            rows = tbody.find_elements(By.TAG_NAME, "tr")
            print(f"Found {len(rows)} rows")

            data = []
            for row in rows:
                try:
                    cells = row.find_elements(By.CSS_SELECTOR, "th, td")
                    row_data = [cell.text for cell in cells]
                    if row_data:
                        data.append(row_data)
                except Exception as e:
                    print(f"Error processing row: {str(e)}")
                    continue

            # Create DataFrame
            print("Creating DataFrame...")
            df = pd.DataFrame(data, columns=headers)
            
            # Save to CSV with error handling
            try:
                print(f"Attempting to save data to: {csv_filename}")
                df.to_csv(csv_filename, index=False)
                print(f"Data successfully saved to {csv_filename}")
                
                # Print first few rows to verify data
                print("\nFirst few rows of data:")
                print(df.head())
                
            except PermissionError:
                # If permission error, try saving to current directory
                fallback_filename = "football_stats_big5_expanded.csv"
                print(f"Permission denied. Trying to save to current directory as: {fallback_filename}")
                df.to_csv(fallback_filename, index=False)
                print(f"Data saved to current directory as: {fallback_filename}")
                
            except Exception as e:
                print(f"Error saving CSV: {str(e)}")
                print("Printing data to console instead:")
                print(df.to_string())
            
        except TimeoutException:
            print("Timeout waiting for elements to load")
        except NoSuchElementException:
            print("Could not find required elements on page")
        except Exception as e:
            print(f"Error during scraping: {str(e)}")
            
    except Exception as e:
        print(f"An error occurred: {str(e)}")
        
    finally:
        print("Closing browser...")
        driver.quit()

if __name__ == "__main__":
    # Try to save in Documents folder
    user_docs = str(Path.home() / "Documents")
    scrape_football_stats(user_docs)

Page loaded successfully
Looking for table container...
Table container found
Looking for switcher...
Successfully clicked switcher
Waiting for table to load...
Table found
Getting headers...
Found 34 headers
Getting data rows...
Found 96 rows
Creating DataFrame...
Attempting to save data to: C:\Users\DATA-JOHN\Documents\football_stats_big5_expanded.csv
Data successfully saved to C:\Users\DATA-JOHN\Documents\football_stats_big5_expanded.csv

First few rows of data:
  ranker         team                comp players_used avg_age possession  \
0      1       Alavés          es La Liga           25    27.2       44.0   
1      2       Angers          fr Ligue 1           23    28.1       42.4   
2      3      Arsenal  eng Premier League           24    26.4       55.2   
3      4  Aston Villa  eng Premier League           24    27.6       49.5   
4      5     Atalanta          it Serie A           28    27.3       56.0   

  games games_starts minutes minutes_90s  ... goals_per90 assists_p